# User Analytics

This notebook adds a simple user-level behavioural analytics layer to the Power BI Usage Intelligence project. The goal is to turn processed usage events into reusable user features and business-friendly user segments.

## 1. Project context

The forecasting pipeline predicts report usage over time. This notebook looks at the same usage ecosystem from a user behaviour lens: who is using reports repeatedly, who has broad engagement, and who appears to be a light or one-time user.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.analytics.user_features import build_user_features
from src.analytics.user_segmentation import build_user_segments

processed_dir = PROJECT_ROOT / "data" / "processed"
metrics_dir = PROJECT_ROOT / "outputs" / "metrics"
segments_dir = PROJECT_ROOT / "outputs" / "segments"

metrics_dir.mkdir(parents=True, exist_ok=True)
segments_dir.mkdir(parents=True, exist_ok=True)

## 2. Load processed input tables

`fact_report_views.csv` is the preferred input because it records report views by user. `dim_user.csv` adds readable user fields, and `dim_date.csv` resolves activity dates from `date_key`.

In [2]:
usage_path = processed_dir / "fact_report_views.csv"
fallback_usage_path = processed_dir / "fact_page_views.csv"

if usage_path.exists():
    usage_events = pd.read_csv(usage_path)
elif fallback_usage_path.exists():
    usage_path = fallback_usage_path
    usage_events = pd.read_csv(usage_path)
else:
    raise FileNotFoundError("No user-level usage fact table was found.")

dim_user = pd.read_csv(processed_dir / "dim_user.csv") if (processed_dir / "dim_user.csv").exists() else pd.DataFrame()
dim_date = pd.read_csv(processed_dir / "dim_date.csv") if (processed_dir / "dim_date.csv").exists() else pd.DataFrame()

print(f"Usage input: {usage_path}")
display(usage_events.head())
display(dim_user.head())

Usage input: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/fact_report_views.csv


,date_key,report_id,user_key,consumption_method,distribution_method,view_count
0,20250101,R_001,UK_0002,Web,Direct,1
1,20250101,R_001,UK_0004,Web,App,3
2,20250101,R_001,UK_0012,Mobile,App,2
3,20250101,R_001,UK_0019,Web,Direct,1
4,20250101,R_001,UK_0023,Web,Direct,1


,user_key,user_id,unique_user
0,UK_0001,user001@masegoinc.com,User 001
1,UK_0002,user002@masegoinc.com,User 002
2,UK_0003,user003@masegoinc.com,User 003
3,UK_0004,user004@masegoinc.com,User 004
4,UK_0005,user005@masegoinc.com,User 005


## 3. Compute user features

The reusable function below creates one row per user with total views, active days, report breadth, first and last activity dates, and a repeat usage flag.

In [3]:
user_features = build_user_features(
    usage_events=usage_events,
    dim_user=dim_user,
    dim_date=dim_date,
)

display(user_features.head(10))

,user_key,user_id,unique_user,total_views,active_days,distinct_reports,avg_views_per_active_day,first_active_date,last_active_date,days_since_last_active,repeat_usage_flag
0,UK_0056,user056@masegoinc.com,User 056,9122,455,30,20.048352,2025-01-01,2026-03-31,0,True
1,UK_0143,user143@masegoinc.com,User 143,8772,455,30,19.279121,2025-01-01,2026-03-31,0,True
2,UK_0141,user141@masegoinc.com,User 141,8560,455,30,18.813187,2025-01-01,2026-03-31,0,True
3,UK_0074,user074@masegoinc.com,User 074,8386,455,30,18.430769,2025-01-01,2026-03-31,0,True
4,UK_0112,user112@masegoinc.com,User 112,8289,455,30,18.217582,2025-01-01,2026-03-31,0,True
5,UK_0168,user168@masegoinc.com,User 168,8128,455,30,17.863736,2025-01-01,2026-03-31,0,True
6,UK_0190,user190@masegoinc.com,User 190,7822,455,30,17.191209,2025-01-01,2026-03-31,0,True
7,UK_0148,user148@masegoinc.com,User 148,7646,455,30,16.804396,2025-01-01,2026-03-31,0,True
8,UK_0152,user152@masegoinc.com,User 152,7633,455,30,16.775824,2025-01-01,2026-03-31,0,True
9,UK_0169,user169@masegoinc.com,User 169,7596,455,30,16.694505,2025-01-01,2026-03-31,0,True


## 4. Review user_features.csv

The feature table is saved as a reusable metric output for reporting, QA, or later modelling work.

In [4]:
user_features_path = metrics_dir / "user_features.csv"
user_features.to_csv(user_features_path, index=False)

print(f"Saved: {user_features_path}")
display(pd.read_csv(user_features_path).head(10))

Saved: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/metrics/user_features.csv


,user_key,user_id,unique_user,total_views,active_days,distinct_reports,avg_views_per_active_day,first_active_date,last_active_date,days_since_last_active,repeat_usage_flag
0,UK_0056,user056@masegoinc.com,User 056,9122,455,30,20.048352,2025-01-01,2026-03-31,0,True
1,UK_0143,user143@masegoinc.com,User 143,8772,455,30,19.279121,2025-01-01,2026-03-31,0,True
2,UK_0141,user141@masegoinc.com,User 141,8560,455,30,18.813187,2025-01-01,2026-03-31,0,True
3,UK_0074,user074@masegoinc.com,User 074,8386,455,30,18.430769,2025-01-01,2026-03-31,0,True
4,UK_0112,user112@masegoinc.com,User 112,8289,455,30,18.217582,2025-01-01,2026-03-31,0,True
5,UK_0168,user168@masegoinc.com,User 168,8128,455,30,17.863736,2025-01-01,2026-03-31,0,True
6,UK_0190,user190@masegoinc.com,User 190,7822,455,30,17.191209,2025-01-01,2026-03-31,0,True
7,UK_0148,user148@masegoinc.com,User 148,7646,455,30,16.804396,2025-01-01,2026-03-31,0,True
8,UK_0152,user152@masegoinc.com,User 152,7633,455,30,16.775824,2025-01-01,2026-03-31,0,True
9,UK_0169,user169@masegoinc.com,User 169,7596,455,30,16.694505,2025-01-01,2026-03-31,0,True


## 5. Create user segments

Segmentation is intentionally rule-based and explainable: one-time users, power users, regular users, and casual users. Quantiles provide adaptive thresholds when fixed business thresholds are not yet available.

In [5]:
user_segments = build_user_segments(user_features)

display(user_segments.head(10))
display(user_segments["user_segment"].value_counts().rename_axis("user_segment").reset_index(name="users"))

,user_key,user_id,unique_user,user_segment,segment_reason
0,UK_0056,user056@masegoinc.com,User 056,power,Total views and distinct reports are both in t...
1,UK_0143,user143@masegoinc.com,User 143,power,Total views and distinct reports are both in t...
2,UK_0141,user141@masegoinc.com,User 141,power,Total views and distinct reports are both in t...
3,UK_0074,user074@masegoinc.com,User 074,power,Total views and distinct reports are both in t...
4,UK_0112,user112@masegoinc.com,User 112,power,Total views and distinct reports are both in t...
5,UK_0168,user168@masegoinc.com,User 168,power,Total views and distinct reports are both in t...
6,UK_0190,user190@masegoinc.com,User 190,power,Total views and distinct reports are both in t...
7,UK_0148,user148@masegoinc.com,User 148,power,Total views and distinct reports are both in t...
8,UK_0152,user152@masegoinc.com,User 152,power,Total views and distinct reports are both in t...
9,UK_0169,user169@masegoinc.com,User 169,power,Total views and distinct reports are both in t...


,user_segment,users
0,regular,139
1,power,50
2,casual,11


## 6. Review user_segments.csv

The segment output keeps the user identifier fields plus the segment label and a plain-English reason.

In [6]:
user_segments_path = segments_dir / "user_segments.csv"
user_segments.to_csv(user_segments_path, index=False)

print(f"Saved: {user_segments_path}")
display(pd.read_csv(user_segments_path).head(10))

Saved: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/segments/user_segments.csv


,user_key,user_id,unique_user,user_segment,segment_reason
0,UK_0056,user056@masegoinc.com,User 056,power,Total views and distinct reports are both in t...
1,UK_0143,user143@masegoinc.com,User 143,power,Total views and distinct reports are both in t...
2,UK_0141,user141@masegoinc.com,User 141,power,Total views and distinct reports are both in t...
3,UK_0074,user074@masegoinc.com,User 074,power,Total views and distinct reports are both in t...
4,UK_0112,user112@masegoinc.com,User 112,power,Total views and distinct reports are both in t...
5,UK_0168,user168@masegoinc.com,User 168,power,Total views and distinct reports are both in t...
6,UK_0190,user190@masegoinc.com,User 190,power,Total views and distinct reports are both in t...
7,UK_0148,user148@masegoinc.com,User 148,power,Total views and distinct reports are both in t...
8,UK_0152,user152@masegoinc.com,User 152,power,Total views and distinct reports are both in t...
9,UK_0169,user169@masegoinc.com,User 169,power,Total views and distinct reports are both in t...


## 7. Save outputs

The required outputs are now available for downstream use:

- `outputs/metrics/user_features.csv`
- `outputs/segments/user_segments.csv`

In [7]:
print("User analytics outputs created")
print(user_features_path)
print(user_segments_path)

User analytics outputs created
/Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/metrics/user_features.csv
/Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/segments/user_segments.csv


## 8. Brief next steps

Useful next extensions include comparing segments by workspace or report category, tracking segment movement over time, and adding stakeholder-friendly summary visuals. Clustering and GenAI are intentionally left out of this first rule-based layer.